In [ ]:
# Yahoo Finance compatibility shim: if yfinance fails (version mismatch such as
# "Impersonating chromeXXX is not supported") or returns nothing, fall back to the Yahoo v8 chart endpoint.
# Prices are dividend/split adjusted, equivalent to auto_adjust=True.
import pandas as pd, requests
try:
    import yfinance as yf
    _yf_download = yf.download
except Exception:
    yf, _yf_download = None, None

def _chart_api(ticker, start, end):
    p1, p2 = int(pd.Timestamp(start).timestamp()), int(pd.Timestamp(end).timestamp())
    for host in ('query1', 'query2'):
        r = requests.get(f'https://{host}.finance.yahoo.com/v8/finance/chart/{ticker}',
                         params=dict(period1=p1, period2=p2, interval='1d', events='div,splits'),
                         headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124 Safari/537.36'}, timeout=30)
        if r.status_code != 200: continue
        res = r.json()['chart']['result'][0]; q = res['indicators']['quote'][0]
        adj = res['indicators'].get('adjclose', [{}])[0].get('adjclose', q['close'])
        d = pd.DataFrame({'Date': pd.to_datetime(res['timestamp'], unit='s'), 'Open': q['open'], 'High': q['high'],
                          'Low': q['low'], 'Close': q['close'], 'Volume': q['volume'], 'Adj': adj}).dropna()
        f = d['Adj'] / d['Close']
        for c in ('Open', 'High', 'Low', 'Close'): d[c] = d[c] * f
        d['Date'] = (d['Date'] + pd.Timedelta(hours=7)).dt.normalize()
        return d.drop(columns='Adj').set_index('Date')
    raise RuntimeError(f'chart API failed for {ticker}')

def _download(ticker, start=None, end=None, **kw):
    try:
        d = _yf_download(ticker, start=start, end=end, **kw) if _yf_download else None
        if d is not None and len(d) > 200: return d
    except Exception:
        pass
    return _chart_api(ticker, start, end)

if yf is not None: yf.download = _download


In [ ]:
# ============================================================================
# SEL 1. KONFIGURASI, ANGGARAN WAKTU COMMIT, DAN CACHE
#   Kaggle memutus commit pada 12 jam dan TIDAK menyimpan apa pun bila terputus.
#   Semua pekerjaan berat di bawah memeriksa anggaran waktu lebih dulu dan
#   dilewati dengan pesan, bukan dengan exception, agar commit selalu ditutup.
#   Setiap pelatihan disimpan ke cache per-item, jadi commit berikutnya
#   melanjutkan dari yang sudah selesai.
# ============================================================================
import os, glob, math, time, pickle, random, warnings, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

T_MULAI       = time.time()
TIME_BUDGET_H = 10.5          # sisakan 1,5 jam dari batas 12 jam
N_SEED        = 15            # ulangan seed untuk evaluasi final
RUN_E4        = True
RUN_E5        = True

CACHE = '/kaggle/working/cache_e45'
os.makedirs(CACHE, exist_ok=True)
for d in glob.glob('/kaggle/input/*/cache_e45') + glob.glob('/kaggle/input/*/*.pkl'):
    try:
        if os.path.isdir(d):
            for f in glob.glob(os.path.join(d, '*.pkl')):
                t = os.path.join(CACHE, os.path.basename(f))
                if not os.path.exists(t): __import__('shutil').copy(f, t)
        else:
            t = os.path.join(CACHE, os.path.basename(d))
            if not os.path.exists(t): __import__('shutil').copy(d, t)
    except Exception:
        pass
print('item cache tersedia:', len(glob.glob(os.path.join(CACHE, '*.pkl'))))

def jam_terpakai(): return (time.time() - T_MULAI) / 3600.0
def sisa_jam():     return TIME_BUDGET_H - jam_terpakai()

DILEWATI = []
def guard(tag, perlu_jam=0.0):
    """True -> lewati. Tidak pernah raise; commit harus selalu selesai."""
    if sisa_jam() <= perlu_jam:
        DILEWATI.append(tag)
        print(f'[skip] {tag}: sisa {sisa_jam():.2f} jam, perlu {perlu_jam:.2f} jam')
        return True
    print(f'[run ] {tag} pada {jam_terpakai():.2f} jam (sisa {sisa_jam():.2f})')
    return False

def disk(key, fn):
    """Cache per-item, penulisan atomik."""
    p = os.path.join(CACHE, key + '.pkl')
    if os.path.exists(p):
        with open(p, 'rb') as f: return pickle.load(f)
    v = fn()
    tmp = p + f'.tmp{os.getpid()}'
    with open(tmp, 'wb') as f: pickle.dump(v, f)
    os.replace(tmp, p)
    return v

def laporan():
    print(f'\nTerpakai {jam_terpakai():.2f} jam dari {TIME_BUDGET_H} jam')
    print('Dilewati commit ini:', DILEWATI if DILEWATI else 'tidak ada')
    print('Isi cache:', len(glob.glob(os.path.join(CACHE, "*.pkl"))), 'item')


In [ ]:
# ============================================================================
# SEL 2. DATA, FITUR, DAN PIPELINE
#
# Versi sebelumnya berhenti dengan "[batal] prasyarat belum ada" karena ia
# memakai fungsi dari notebook. Versi ini membangun ulang seluruh pipeline
# sendiri: memuat SIDO, menyusun 16 fitur, melatih LSTM, dan mengevaluasi.
# Jadi bisa dijalankan di notebook kosong sekalipun.
#
# Pertanyaannya: mengapa naskah melaporkan MAPE 1,66 sedangkan run arsip
# memberi 3,50? Dua dugaan diuji di sini.
#   (a) Konfigurasi PSO terpilih menempel di tepi ruang pencarian:
#       160 unit, dropout 0,1, lr 1e-3, batch 16 — empat dari tujuh
#       hiperparameter persis di batas rentang yang dideklarasikan naskah.
#   (b) Ragam antar-seed besar: sd 0,88 pada run arsip melawan 0,19 di naskah.
# Catatan: train_eval di notebook SUDAH memakai EarlyStopping(patience=10),
# jadi "100 epoch tanpa early stopping" bukan penyebabnya. Protokol di sini
# menyalin protokol itu apa adanya.
#
# Kebutuhan: GPU menyala. Internet ON kalau ingin fitur eksternal JCI dan
# USD/IDR; tanpa internet, kedua kolom diisi nol dan itu DICETAK sebagai
# peringatan karena membuat angkanya tidak sebanding dengan run arsip.
# Waktu: ~35 menit dengan T4 (4 konfigurasi x 15 seed).
# =====================================================================
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam, RMSprop, Nadam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)

TARGET, TRAIN_FRAC = 'Close', 0.8
START, END = '2019-07-01', '2025-07-01'
N_SEED = 15
FINAL_EPOCHS = 100
_gpu = tf.config.list_physical_devices('GPU')
print('GPU:', _gpu)
if not _gpu:
    print('PERINGATAN: GPU tidak terdeteksi, seluruh pelatihan berjalan di CPU.\n'
          '  4 konfigurasi x 15 seed akan memakan sekitar 3-4 jam, bukan 35 menit.\n'
          '  Nyalakan Accelerator GPU T4 x2 di panel kanan, lalu restart sesi.\n'
          '  Kalau tetap ingin lanjut di CPU, turunkan N_SEED menjadi 5.')

# ---------- 1. data SIDO ----------------------------------------------
def _cari(nama_pola):
    for pola in (f'/kaggle/input/**/{nama_pola}', f'./{nama_pola}',
                 f'/kaggle/working/**/{nama_pola}'):
        hit = glob.glob(pola, recursive=True)
        if hit: return hit[0]
    return None

def muat_sido():
    f = _cari('SIDO_Stock_Data.xlsx') or _cari('SIDO*.xlsx') or _cari('SIDO*.csv')
    if f:
        print('sumber harga:', f)
        d = pd.read_excel(f) if f.endswith(('.xlsx', '.xls')) else pd.read_csv(f)
        d.columns = [str(c).strip().title() for c in d.columns]
        if 'Date' not in d.columns: d = d.rename(columns={d.columns[0]: 'Date'})
        d['Date'] = pd.to_datetime(d['Date'])
        return d.sort_values('Date').reset_index(drop=True)
    import yfinance as yf
    print('sumber harga: yfinance SIDO.JK')
    d = yf.download('SIDO.JK', start=START, end=END, auto_adjust=True, progress=False)
    if isinstance(d.columns, pd.MultiIndex): d.columns = d.columns.get_level_values(0)
    return d.reset_index().rename(columns={'index': 'Date'})

raw = muat_sido()
if 'Close' not in raw.columns:
    raise SystemExit('kolom Close tidak ada di data harga')

PUNYA_OHLCV = all(c in raw.columns for c in ('Open', 'High', 'Low', 'Close', 'Volume'))
if not PUNYA_OHLCV:
    # Berkas SIDO_Stock_Data.xlsx hanya memuat Date dan Close (1449 baris).
    # Coba lengkapi dari yfinance; kalau gagal, lanjut dengan Close saja dan
    # katakan apa adanya bahwa angkanya hanya sebanding dengan baris
    # "Close only" pada feature_ablation.csv (2,25%), bukan dengan 16 fitur.
    try:
        import yfinance as yf
        o = yf.download('SIDO.JK', start=START, end=END, auto_adjust=True, progress=False)
        if isinstance(o.columns, pd.MultiIndex): o.columns = o.columns.get_level_values(0)
        o = o.reset_index()[['Date','Open','High','Low','Close','Volume']]
        raw = o.sort_values('Date').reset_index(drop=True)
        PUNYA_OHLCV = True
        print('OHLCV dilengkapi dari yfinance')
    except Exception as ex:
        print(f'PERINGATAN: OHLCV tidak tersedia ({type(ex).__name__}); '
              f'hanya Close yang dipakai.')

# ---------- 2. fitur: salinan persis dari notebook ----------------------
def _ema(s, span): return s.ewm(span=span, adjust=False).mean()

def add_technical(df):
    d = df.copy(); c = d.Close
    d['SMA10'] = c.rolling(10).mean(); d['SMA20'] = c.rolling(20).mean()
    d['EMA10'] = _ema(c, 10); d['EMA20'] = _ema(c, 20)
    delta = c.diff(); up = delta.clip(lower=0); dn = -delta.clip(upper=0)
    rs = up.ewm(alpha=1/14, adjust=False).mean() / dn.ewm(alpha=1/14, adjust=False).mean()
    d['RSI14'] = 100 - 100/(1 + rs)
    macd = _ema(c, 12) - _ema(c, 26); d['MACD'] = macd; d['MACD_sig'] = _ema(macd, 9)
    d['ret1'] = c.pct_change(); d['vol10'] = d['ret1'].rolling(10).std()
    return d

def attach_external(df):
    d = df.copy(); hilang = []
    for name, tk in {'JKSE': '^JKSE', 'USDIDR': 'USDIDR=X'}.items():
        try:
            import yfinance as yf
            e = yf.download(tk, start=START, end=END, auto_adjust=True, progress=False)
            if isinstance(e.columns, pd.MultiIndex): e.columns = e.columns.get_level_values(0)
            e = e.reset_index()[['Date', 'Close']].rename(columns={'Close': name})
            d = d.merge(e, on='Date', how='left'); d[name] = d[name].ffill().bfill()
        except Exception as ex:
            d[name] = 0.0; hilang.append(name)
    if hilang:
        print(f'PERINGATAN: fitur eksternal {hilang} diisi nol (internet mati). '
              f'Angka di bawah TIDAK sebanding dengan run arsip 16 fitur.')
    return d

df_primary = attach_external(add_technical(raw)).dropna().reset_index(drop=True)
OHLCV = ['Open', 'High', 'Low', 'Close', 'Volume'] if PUNYA_OHLCV else ['Close']
TA    = ['SMA10','SMA20','EMA10','EMA20','RSI14','MACD','MACD_sig','ret1','vol10']
EXT   = [c for c in ('JKSE', 'USDIDR') if df_primary[c].abs().sum() > 0]
DEFAULT_FEATURES = OHLCV + TA + EXT
SET_NAME = ('OHLCV' if PUNYA_OHLCV else 'Close') + '+TA' + ('+EXT' if EXT else '')
print(f'baris: {len(df_primary)} | fitur: {len(DEFAULT_FEATURES)} ({SET_NAME}) | '
      f'{df_primary.Date.min().date()} -> {df_primary.Date.max().date()}')
_ACUAN = {'Close+TA+EXT': 2.25, 'Close+TA': 2.25,
          'OHLCV+TA': 2.21, 'OHLCV+TA+EXT': 3.50}
print(f"acuan feature_ablation.csv untuk set ini: "
      f"{_ACUAN.get(SET_NAME, 'tidak ada pembanding langsung')}")

# ---------- 3. windowing, model, pelatihan: protokol notebook -----------
def make_sequences(X, y, timestep):
    Xs, ys = [], []
    for i in range(timestep, len(X)):
        Xs.append(X[i-timestep:i]); ys.append(y[i])
    return np.array(Xs), np.array(ys)

def prepare_data(df, cols, timestep):
    dX = df[cols].values.astype('float32'); dy = df[[TARGET]].values.astype('float32')
    n = len(df); n_tr = int(n * TRAIN_FRAC)
    sx = MinMaxScaler().fit(dX[:n_tr]); sy = MinMaxScaler().fit(dy[:n_tr])
    Xseq, yseq = make_sequences(sx.transform(dX), sy.transform(dy), timestep)
    sp = n_tr - timestep
    return Xseq[:sp], yseq[:sp], Xseq[sp:], yseq[sp:], sy

def _opt(name, lr): return {'adam': Adam, 'rmsprop': RMSprop, 'nadam': Nadam}[name](lr)

def build_lstm(shape, units, n_layers, dropout, lr, optimizer):
    K.clear_session()
    inp = Input(shape=shape); x = inp
    for i in range(int(n_layers)):
        x = LSTM(int(units), return_sequences=(i < int(n_layers)-1))(x)
        if dropout > 0: x = Dropout(dropout)(x)
    m = Model(inp, Dense(1, activation='linear')(x))
    m.compile(loss='mse', optimizer=_opt(optimizer, lr))
    return m

def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true).flatten(); y_pred = np.asarray(y_pred).flatten()
    return {'MAPE': float(np.mean(np.abs((y_true-y_pred)/y_true))*100),
            'RMSE': math.sqrt(np.mean((y_true-y_pred)**2)),
            'MAE':  float(np.mean(np.abs(y_true-y_pred)))}

def train_eval(hp, seed):
    set_seed(seed)
    Xtr, ytr, Xte, yte, sy = prepare_data(df_primary, DEFAULT_FEATURES, int(hp['timestep']))
    m = build_lstm(Xtr.shape[1:], hp['units'], hp.get('n_layers', 1),
                   hp.get('dropout', 0.2), hp.get('lr', 1e-3), hp.get('optimizer', 'adam'))
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    m.fit(Xtr, ytr, validation_split=0.2, shuffle=False, epochs=FINAL_EPOCHS,
          batch_size=int(hp.get('batch_size', 32)), callbacks=[es], verbose=0)
    p = m.predict(Xte, verbose=0)
    return evaluate(sy.inverse_transform(yte), sy.inverse_transform(p))

# ---------- 4. baseline naif -------------------------------------------
def naive_mape():
    n = len(df_primary); n_tr = int(n * TRAIN_FRAC)
    y = df_primary[TARGET].values.astype('float64')
    yt = y[n_tr:]; yp = y[n_tr - 1:-1]
    return float(np.mean(np.abs((yt - yp) / yt)) * 100)

NAIVE = naive_mape()
print(f'MAPE random walk: {NAIVE:.4f}')



In [ ]:
# ============================================================================
# SEL 3. E4 — konfigurasi arsip vs interior, 15 seed, protokol final identik
# ============================================================================
# ---------- 5. konfigurasi yang dibandingkan ----------------------------
PSO_HP = dict(units=160, n_layers=1, dropout=0.1,  lr=1e-3,    batch_size=16, timestep=21, optimizer='nadam')
GA_HP  = dict(units=126, n_layers=1, dropout=0.140, lr=5.261e-4, batch_size=61, timestep=22, optimizer='rmsprop')
STD_HP = dict(units=64,  n_layers=1, dropout=0.2,  lr=1e-3,    batch_size=32, timestep=20, optimizer='adam')
INT_HP = dict(units=88,  n_layers=1, dropout=0.3,  lr=3.2e-4,  batch_size=64, timestep=20, optimizer='adam')
BOUNDS = dict(units=(16,160), dropout=(0.1,0.5), lr=(1e-4,1e-3), batch_size=(16,128), timestep=(10,30))

def di_tepi(hp):
    return ','.join(k for k,(lo,hi) in BOUNDS.items()
                    if abs(float(hp[k])-lo) < 1e-9 or abs(float(hp[k])-hi) < 1e-9) or 'none'

rows = []
if RUN_E4 and not guard('E4 (4 konfigurasi x %d seed)' % N_SEED, perlu_jam=0.8):
  for lbl, hp in (('LSTM-PSO (arsip)', PSO_HP), ('LSTM-GA (arsip)', GA_HP),
                ('LSTM (standard)', STD_HP), ('LSTM (interior)', INT_HP)):
    mapes = []
    for rep in range(N_SEED):
        if sisa_jam() < 0.3:
            print(f'  [stop] anggaran habis saat {lbl} seed {rep}'); break
        mapes.append(disk(f'e4_{lbl}_s{rep}',
                          lambda hp=hp, rep=rep: train_eval(hp, SEED + rep))['MAPE'])
    if not mapes: continue
    a = np.array(mapes, float)
    rows.append(dict(config=lbl, feature_set=SET_NAME, n_seed=len(a),
                     MAPE_mean=round(a.mean(), 3), MAPE_sd=round(a.std(ddof=1), 3),
                     MAPE_min=round(a.min(), 3), MAPE_max=round(a.max(), 3),
                     share_below_2pct=round(float((a < 2.0).mean()), 2),
                     vs_naive=round(a.mean()/NAIVE, 2), at_bounds=di_tepi(hp)))
    print(f"  {lbl:18s} MAPE {a.mean():.3f} (sd {a.std(ddof=1):.3f}) "
          f"min {a.min():.3f} max {a.max():.3f} | di tepi: {di_tepi(hp)}")

E4 = pd.DataFrame(rows)
if len(E4) == 0:
    print('E4 dilewati seluruhnya pada commit ini; tidak ada tabel yang ditulis.')
else:
    E4.to_csv('/kaggle/working/TableE4_config_vs_seed.csv', index=False)
    print('\n' + E4.to_string(index=False))

    fig, ax = plt.subplots(figsize=(7.5, 4))
    ax.bar(E4.config, E4.MAPE_mean,
           yerr=[E4.MAPE_mean - E4.MAPE_min, E4.MAPE_max - E4.MAPE_mean],
           color='0.6', capsize=4)
    ax.axhline(NAIVE, color='k', lw=1.2, label=f'Random walk ({NAIVE:.2f})')
    ax.axhline(1.66, color='0.3', lw=1, ls='--', label='Naskah lama (1.66)')
    ax.set_ylabel('Test MAPE (%)'); ax.legend(frameon=False, fontsize=8)
    ax.grid(axis='y', alpha=.3); plt.xticks(rotation=12, fontsize=8)
    fig.tight_layout()
    fig.savefig('/kaggle/working/FigE4_config_vs_seed.png', dpi=600)
    plt.show()

    print('\nCara membaca E4:')
    print(' - Konfigurasi interior lebih baik dan jauh lebih stabil daripada')
    print('   konfigurasi di tepi ruang -> pencarian terperangkap di batas.')
    print(' - Tidak ada seed yang mencapai 1,66 -> angka naskah lama tidak')
    print('   dapat direproduksi dari konfigurasi ini pada protokol ini.')


In [ ]:
# ============================================================================
# SEL 4. E5 — definisi ruang diperketat, fitness, GA dan PSO
# ============================================================================
import itertools
# ---------------- ruang pencarian yang diperketat ----------------------
# Batas lama (naskah): units 16-160, dropout 0.1-0.5, lr 1e-4-1e-3, batch 16-128.
# Batas baru menjauh dari wilayah berkapasitas besar dan beregularisasi rendah
# yang terbukti rapuh di E4. Dipilih SEBELUM melihat hasil pencarian baru.
TIGHT = dict(units=(32, 96), n_layers=(1, 2), dropout=(0.20, 0.45),
             lr=(1e-4, 5e-4), batch_size=(32, 96), timestep=(15, 25))
OPTS  = ['adam', 'rmsprop', 'nadam']
LB = np.array([TIGHT['units'][0], TIGHT['n_layers'][0], TIGHT['dropout'][0],
               math.log10(TIGHT['lr'][0]), TIGHT['batch_size'][0],
               TIGHT['timestep'][0], 0], float)
UB = np.array([TIGHT['units'][1], TIGHT['n_layers'][1], TIGHT['dropout'][1],
               math.log10(TIGHT['lr'][1]), TIGHT['batch_size'][1],
               TIGHT['timestep'][1], len(OPTS) - 1e-9], float)

def vec_to_hp(v):
    return dict(units=int(round(v[0])), n_layers=int(round(v[1])),
                dropout=round(float(v[2]), 3), lr=float(10 ** v[3]),
                batch_size=int(round(v[4])), timestep=int(round(v[5])),
                optimizer=OPTS[int(min(int(v[6]), len(OPTS) - 1))])

def cfg_key(v):
    h = vec_to_hp(v)
    return (h['units'], h['n_layers'], round(h['dropout'], 2),
            round(math.log10(h['lr']), 2), h['batch_size'], h['timestep'],
            h['optimizer'])

# ---------------- anggaran dan cache -----------------------------------
BUDGET      = 100      # evaluasi kandidat BERBEDA per optimizer
POP         = 20
FIT_EPOCHS  = 20
N_SEED_FIN  = 15
TIME_CAP_H  = 2.0      # berhenti mencari bila melewati ini
E5DIR = CACHE; os.makedirs(E5DIR, exist_ok=True)
def _habis(): return sisa_jam() < 2.0   # sisakan 2 jam untuk evaluasi final

# ---------------- fitness: validation MSE, bukan test ------------------
# Validasi diambil dari 20 persen terakhir porsi latih, jadi data uji tidak
# pernah tersentuh selama pencarian. Ini sekaligus menjawab keberatan
# reviewer soal pemakaian data uji dalam seleksi model.
def fitness(v, seed=SEED):
    hp = vec_to_hp(v)
    def _hit():
        set_seed(seed)
        Xtr, ytr, _, _, _ = prepare_data(df_primary, DEFAULT_FEATURES, hp['timestep'])
        n_val = max(1, int(0.2 * len(Xtr)))
        Xa, ya, Xv, yv = Xtr[:-n_val], ytr[:-n_val], Xtr[-n_val:], ytr[-n_val:]
        m = build_lstm(Xa.shape[1:], hp['units'], hp['n_layers'], hp['dropout'],
                       hp['lr'], hp['optimizer'])
        m.fit(Xa, ya, validation_data=(Xv, yv), epochs=FIT_EPOCHS, shuffle=False,
              batch_size=hp['batch_size'], verbose=0)
        return float(np.mean((m.predict(Xv, verbose=0).ravel() - yv.ravel()) ** 2))
    return disk('fit_' + '_'.join(map(str, cfg_key(v))), _hit)

_MEMO = {}
def fit_counted(v):
    k = cfg_key(v)
    if k in _MEMO: return _MEMO[k], False
    f = fitness(v); _MEMO[k] = f
    return f, True

# ---------------- GA ---------------------------------------------------
def run_ga(seed=SEED, budget=BUDGET, pop=POP):
    rng = np.random.RandomState(seed)
    X = rng.uniform(LB, UB, size=(pop, len(LB)))
    f = np.array([fit_counted(x)[0] for x in X]); spent = len(_MEMO)
    best, bestf, curve = X[f.argmin()].copy(), f.min(), [f.min()]
    n_elite = max(2, pop // 4)
    while spent < budget and not _habis():
        order = np.argsort(f); newX = [X[i].copy() for i in order[:n_elite]]
        while len(newX) < pop:
            a, b = [X[rng.choice(order[:pop // 2])] for _ in range(2)]
            cx = rng.rand(len(LB)) < 0.5
            ch = np.where(cx, a, b)
            mut = rng.rand(len(LB)) < 0.2
            ch = np.where(mut, ch + rng.normal(0, 0.1, len(LB)) * (UB - LB), ch)
            newX.append(np.clip(ch, LB, UB))
        X = np.array(newX)
        f = np.array([fit_counted(x)[0] for x in X]); spent = len(_MEMO)
        if f.min() < bestf: bestf, best = f.min(), X[f.argmin()].copy()
        curve.append(bestf)
        print(f'  [GA-tight] distinct={spent:3d}/{budget}  best val MSE={bestf:.8f}')
    return best, float(bestf), np.array(curve, float), spent

# ---------------- PSO, dengan penjaga kemandekan -----------------------
def run_pso(seed=SEED, budget=BUDGET, swarm=POP, w=0.729, c1=1.49445, c2=1.49445):
    rng = np.random.RandomState(seed)
    X = rng.uniform(LB, UB, size=(swarm, len(LB))); V = np.zeros_like(X)
    pb, pbf = X.copy(), np.full(swarm, np.inf)
    gb, gbf, curve = None, np.inf, []
    base = len(_MEMO); stall = 0; it = 0
    for i in range(swarm):
        f, _ = fit_counted(X[i]); pbf[i] = f
        if f < gbf: gbf, gb = f, X[i].copy()
    curve.append(gbf)
    while len(_MEMO) - base < budget and it < 500 and not _habis():
        it += 1; before = len(_MEMO)
        r1, r2 = rng.rand(*X.shape), rng.rand(*X.shape)
        V = w * V + c1 * r1 * (pb - X) + c2 * r2 * (gb - X)
        X = np.clip(X + V, LB, UB)
        for i in range(swarm):
            f, _ = fit_counted(X[i])
            if f < pbf[i]: pbf[i], pb[i] = f, X[i].copy()
            if f < gbf:    gbf, gb = f, X[i].copy()
        curve.append(gbf)
        if len(_MEMO) == before:
            stall += 1
            if stall >= 15:        # swarm mengerumun: suntik keragaman
                k = max(1, swarm // 4); worst = np.argsort(pbf)[-k:]
                X[worst] = rng.uniform(LB, UB, size=(k, len(LB)))
                V[worst] = 0.0; pbf[worst] = np.inf; stall = 0
        else:
            stall = 0
        print(f'  [PSO-tight] distinct={len(_MEMO)-base:3d}/{budget} '
              f'iter={it} best val MSE={gbf:.8f}')
    return gb, float(gbf), np.array(curve, float), len(_MEMO) - base



In [ ]:
# ============================================================================
# SEL 5. E5 — jalankan pencarian (dibatasi anggaran commit)
# ============================================================================
PSO_T = GA_T = None
pso_f = ga_f = float('nan'); pso_n = ga_n = 0
if RUN_E5 and not guard('E5 pencarian ruang diperketat', perlu_jam=2.5):
    # ---------------- jalankan pencarian -----------------------------------
    print('=== pencarian pada ruang diperketat ===')
    print('batas:', {k: v for k, v in TIGHT.items()})
    _MEMO = {}
    pso_v, pso_f, pso_c, pso_n = disk('search_pso_tight', run_pso)
    _MEMO = {}
    ga_v, ga_f, ga_c, ga_n = disk('search_ga_tight', run_ga)
    PSO_T = vec_to_hp(pso_v); GA_T = vec_to_hp(ga_v)
    print('\nPSO-tight:', PSO_T, f'| val MSE {pso_f:.8f} | {pso_n} evaluasi berbeda')
    print('GA-tight :', GA_T,  f'| val MSE {ga_f:.8f} | {ga_n} evaluasi berbeda')

    def di_tepi(hp):
        tepi = []
        for k, (lo, hi) in TIGHT.items():
            v = float(hp[k])
            if abs(v - lo) < 1e-9 or abs(v - hi) < 1e-9: tepi.append(k)
        return ','.join(tepi) or 'none'



In [ ]:
# ============================================================================
# SEL 6. E5 — evaluasi final, tabel, gambar, laporan waktu
# ============================================================================
# ---------------- evaluasi final, protokol identik E4 ------------------
PSO_ARSIP = dict(units=160, n_layers=1, dropout=0.1, lr=1e-3,
                 batch_size=16, timestep=21, optimizer='nadam')
INTERIOR  = dict(units=88, n_layers=1, dropout=0.3, lr=3.2e-4,
                 batch_size=64, timestep=20, optimizer='adam')

kandidat = [('LSTM (interior)', INTERIOR), ('LSTM-PSO (arsip)', PSO_ARSIP)]
if PSO_T: kandidat.insert(0, ('PSO-tight', PSO_T))
if GA_T:  kandidat.insert(1, ('GA-tight', GA_T))

rows = []
for lbl, hp in kandidat:
    vals = []
    for r in range(N_SEED_FIN):
        if sisa_jam() < 0.25:
            print(f'  [stop] anggaran habis saat {lbl} seed {r}'); break
        vals.append(disk(f'e5_final_{lbl}_s{r}',
                         lambda hp=hp, r=r: train_eval(hp, SEED + r))['MAPE'])
    if not vals: continue
    a = np.array(vals, float)
    rows.append(dict(config=lbl, n_seed=len(a),
                     MAPE_mean=round(a.mean(), 3), MAPE_sd=round(a.std(ddof=1), 3),
                     MAPE_min=round(a.min(), 3), MAPE_max=round(a.max(), 3),
                     share_below_2pct=round(float((a < 2.0).mean()), 2),
                     vs_naive=round(a.mean() / NAIVE, 2), at_bounds=di_tepi(hp)))
    print(f"  {lbl:18s} MAPE {a.mean():.3f} (sd {a.std(ddof=1):.3f}) "
          f"min {a.min():.3f} max {a.max():.3f}")

E5 = pd.DataFrame(rows)
if len(E5) == 0:
    print('E5 dilewati pada commit ini; jalankan commit lanjutan.')
else:
  E5.to_csv('/kaggle/working/TableE5_tight_space.csv', index=False)
  print('\n' + E5.to_string(index=False))

  fig, ax = plt.subplots(figsize=(7.5, 4))
  ax.bar(E5.config, E5.MAPE_mean,
         yerr=[E5.MAPE_mean - E5.MAPE_min, E5.MAPE_max - E5.MAPE_mean],
         color='0.6', capsize=4)
  ax.axhline(NAIVE, color='k', lw=1.2, label=f'Random walk ({NAIVE:.2f})')
  ax.set_ylabel('Test MAPE (%)'); ax.legend(frameon=False, fontsize=8)
  ax.grid(axis='y', alpha=.3); plt.xticks(rotation=12, fontsize=8)
  fig.tight_layout(); fig.savefig('/kaggle/working/FigE5_tight_space.png', dpi=600)
  plt.show()

print('\nCara membaca:')
print(' - PSO-tight atau GA-tight mengalahkan interior -> kegagalan di E4 milik')
print('   RUANG pencarian, bukan metaheuristiknya. Laporkan batas ruang sebagai')
print('   faktor desain, dan sebutkan bahwa batas lama memuat wilayah rapuh.')
print(' - Keduanya tetap kalah dari interior -> pada n sekecil ini pencarian')
print('   hiperparameter tidak membayar dirinya sendiri. Ini kesimpulan negatif')
print('   yang kuat dan langsung menjawab Reviewer B.')
print(' - Bandingkan juga sd: konfigurasi yang stabil lebih berguna bagi praktisi')
print('   daripada konfigurasi dengan rata-rata sedikit lebih baik tetapi rapuh.')

laporan()
